# SpeechT5 + WavLM Fine-Tuning

This notebook fine-tunes **SpeechT5** (`microsoft/speecht5_vc`) to accept **WavLM hidden states**
as encoder input instead of raw audio, enabling a WavLM-based feature-space speech translation pipeline.

## Architecture
```
EN raw audio  ──► [WavLM Encoder (frozen)]  ──► EN hidden states (Seq, 768)
                                                        │
                                 [SpeechT5 Transformer (fine-tuned)] ──► DE mel spectrogram
                                                        │
                                             [HiFi-GAN Vocoder]  ──► DE audio
```

## Required Pre-Step
The preprocessed WavLM dataset must exist at:
```
datasets/processed_wavlm_en_de_v1/
    en/   ← WavLM hidden states (Seq, 768)
    de/   ← WavLM hidden states (Seq, 768)
```
Run `preprocess_wavlm.py` if the dataset is not yet generated.

> **Note on the target representation:**  
> The dataset stores WavLM features for both sides. The `SpeechT5WavLMDataset`
> class in `model.py` handles the fallback to mel-spectrogram extraction if the
> target is raw audio. The fine-tuning loss is computed against SpeechT5's
> mel-spectrogram decoder output — this is what teaches the model to bridge
> the WavLM representation into something the trained decoder can process.


## 1. Setup & Imports

In [ ]:
import os
import sys

# Add project root to Python path so that dataset_loader and encoders are importable.
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.insert(0, project_root)

from model import SpeechT5WavLM
import dataset_loader
from datasets import load_from_disk
import numpy as np
from IPython.display import Audio, display

print(f"Project root: {project_root}")
print(f"Datasets directory: {dataset_loader.DATASETS_DIR}")

## 2. Configuration

Edit the constants below to control training behaviour.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Path to the preprocessed WavLM dataset produced by preprocess_wavlm.py.
# ─────────────────────────────────────────────────────────────────────────────
PREPROCESSED_PATH = os.path.join(dataset_loader.DATASETS_DIR, "processed_wavlm_en_de_v8")

# ─────────────────────────────────────────────────────────────────────────────
# Training hyper-parameters
# ─────────────────────────────────────────────────────────────────────────────
EPOCHS         = 10     # Total training epochs (Fine Phase)
COARSE_EPOCHS  = 4     # Epochs for Phase A (mel targets strided 2x for coarser supervision; config.reduction_factor stays at 2 throughout).
BATCH_SIZE     = 1       # Keep small — WavLM sequences are variable length
LEARNING_RATE  = 1e-4    # Learning rate for fine-tuning

# ─────────────────────────────────────────────────────────────────────────────
# Output checkpoint directory
# ─────────────────────────────────────────────────────────────────────────────
CHECKPOINT_DIR = "speecht5_wavlm_en_de_v2026_05_15_002"
SAVE_EVERY     = 10

print(f"Preprocessed data: {PREPROCESSED_PATH}")
print(f"Total Epochs: {EPOCHS + COARSE_EPOCHS} (Coarse: {COARSE_EPOCHS}, Fine: {EPOCHS})")
print(f"Batch size: {BATCH_SIZE}  |  LR: {LEARNING_RATE}")
print(f"Checkpoint will be saved to: {CHECKPOINT_DIR}")

## 4. Initialise the Model

`SpeechT5WavLM` loads:
- **SpeechT5** (`microsoft/speecht5_vc`) — the transformer to fine-tune
- **HiFi-GAN** (`microsoft/speecht5_hifigan`) — the vocoder (frozen, used at inference)

WavLM itself is **not** loaded here; it was already used during preprocessing to produce the
hidden states stored in the dataset. It is only needed again at inference time
(handled by `run_inference`).

In [ ]:
model = SpeechT5WavLM()
model.load("speecht5_wavlm_en_de_v2026_05_15_001")
print(f"\nModel loaded on device: {model.device}")

## 4.5 Overfitting Test (Single Sample)

This cell takes exactly one sample from the preprocessed dataset and runs many epochs to ensure the model can achieve near-zero loss on it. This is a critical diagnostic step to verify that the architecture is capable of memorizing data before running a full training session.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
import os
import shutil
from datasets import load_from_disk
import numpy as np
import torch

# --- 1. Configuration & Data Preparation ---
OVERFIT_TEMP_DIR      = "temp_overfit_dataset"
OVERFIT_TARGET_LOSS   = 0.05   # Model must reach this loss to pass
OVERFIT_MAX_EPOCHS    = 200    # Hard cap
OVERFIT_COARSE_EPOCHS = 40

if os.path.exists(OVERFIT_TEMP_DIR):
    shutil.rmtree(OVERFIT_TEMP_DIR)

print(f"Loading dataset from {PREPROCESSED_PATH}...")
full_dataset = load_from_disk(PREPROCESSED_PATH)
# Select exactly one sample for overfitting
single_sample = full_dataset.select([1])
single_sample.save_to_disk(OVERFIT_TEMP_DIR)

# --- 2. Visualization Callback (also accumulates losses for the gate) ---
overfit_losses = []

def overfit_viz_callback(step, loss, target_mel, pred_mel):
    overfit_losses.append(loss)
    if step % 50 == 0 or step == 1:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))

        # Target Spectrogram
        t_mel = target_mel[0].cpu().numpy()
        t_mel[t_mel == -100] = np.nan
        im1 = axes[0].imshow(t_mel.T, origin='lower', aspect='auto')
        axes[0].set_title(f"Target Mel-spectrogram (Step {step})")
        plt.colorbar(im1, ax=axes[0])

        # Predicted Spectrogram
        p_mel = pred_mel[0].detach().cpu().numpy()
        im2 = axes[1].imshow(p_mel.T, origin='lower', aspect='auto')
        axes[1].set_title(f"Predicted Mel-spectrogram (Step {step}, Loss: {loss:.4f})")
        plt.colorbar(im2, ax=axes[1])

        plt.tight_layout()
        plt.show()
        plt.close(fig)

# --- 3. Run Overfitting ---
print("Starting Overfitting Test with Real-time Visualization...")

model.fine_tune(
    preprocessed_path=OVERFIT_TEMP_DIR,
    epochs=OVERFIT_MAX_EPOCHS,
    learning_rate=1e-4,
    batch_size=1,
    saving_checkpoint=99999,  # Don't save diagnostic checkpoints
    coarse_epochs=OVERFIT_COARSE_EPOCHS,
    step_callback=overfit_viz_callback
)

# --- 4. Quantitative Pass Gate ---
# Do not proceed to section 6 unless this assertion passes.
overfit_final_loss = overfit_losses[-1] if overfit_losses else float('inf')
assert overfit_final_loss < OVERFIT_TARGET_LOSS, (
    f"Overfitting test FAILED: final loss {overfit_final_loss:.4f} did not reach "
    f"target {OVERFIT_TARGET_LOSS}. Diagnose the architecture before proceeding to full training."
)
print(f"\n✓ Overfitting test PASSED: final loss = {overfit_final_loss:.4f}")


## 5. Extract Target Speaker Embedding

The SpeechT5 decoder needs an **x-vector speaker embedding** to condition its output voice.

`get_speaker_embedding` streams one sample from Google FLEURS for the target language and
computes the x-vector using the SpeechBrain classifier. The embedding is saved alongside
the model checkpoint.

In [ ]:
model.get_speaker_embedding(target_lang="de")

## 6. Fine-Tune

Training will:
1. Load the preprocessed WavLM dataset.
2. Freeze SpeechT5's **encoder prenet** (bypassed by WavLM feature injection anyway).
   All other sub-modules — transformer encoder, decoder, postnet, Conv1DBridge, and
   `encoder_spk_proj` — are trained.
3. Feed WavLM hidden states through the Conv1DBridge and directly into SpeechT5's
   transformer encoder, bypassing the encoder prenet.
4. Compute the spectrogram loss against the decoder's mel output.
5. Save a checkpoint every `saving_checkpoint` epochs. On `KeyboardInterrupt`
   progress is saved automatically to `speecht5_wavlm_interrupted`.

> **Prerequisite:** The overfitting test in section 4.5 must have printed
> `✓ Overfitting test PASSED` before running this cell. If it did not,
> diagnose the architecture before proceeding.


In [ ]:
# ── Dataset scale inspection ──────────────────────────────────────────────
# Run this before full training to verify all mel spectrograms are in the
# expected log-mel range (~[-5, 0]). Values like 0–500 indicate a
# preprocessing scale mismatch that will cause loss ≫ 1.
from datasets import load_from_disk
import numpy as np

ds = load_from_disk(PREPROCESSED_PATH)
print(f"Dataset size: {len(ds)} samples\n")
for i in [0, 1, 10, 50, 100]:
    if i >= len(ds):
        break
    row = ds[i]
    tgt = np.array(row["labels"], dtype=np.float32).reshape(-1, 80)
    print(f"Row {i:>3}: mel  min={tgt.min():.2f}  max={tgt.max():.2f}  mean={tgt.mean():.2f}  shape={tgt.shape}")

print("\nExpected: min ≈ -5 to -8, max ≈ 0 to 2. If values are wildly different, rerun preprocessing.")


In [ ]:
# ── Overfitting gate ─────────────────────────────────────────────────────────
# This will raise AssertionError if section 4.5 was skipped or failed.
# assert 'overfit_final_loss' in dir() and overfit_final_loss < OVERFIT_TARGET_LOSS, \
#     "Overfitting test has not been run or did not pass. Run section 4.5 first."

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

def viz_callback(step, loss, target_mel, pred_mel):
    if step % 50 == 0:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        
        # Target Spectrogram
        t_mel = target_mel[0].cpu().numpy()
        t_mel[t_mel == -100] = np.nan
        im1 = axes[0].imshow(t_mel.T, origin='lower', aspect='auto')
        axes[0].set_title(f"Ground Truth (Step {step})")
        plt.colorbar(im1, ax=axes[0])
        
        # Predicted Spectrogram
        p_mel = pred_mel[0].detach().cpu().numpy()
        im2 = axes[1].imshow(p_mel.T, origin='lower', aspect='auto')
        axes[1].set_title(f"Predicted (Step {step}, Loss: {loss:.4f})")
        plt.colorbar(im2, ax=axes[1])
        
        plt.tight_layout()
        plt.show()
        plt.close(fig)

print(f"Starting Coarse-to-Fine Training...")

model.fine_tune(
    preprocessed_path=PREPROCESSED_PATH,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    saving_checkpoint=SAVE_EVERY,
    coarse_epochs=COARSE_EPOCHS,
    step_callback=viz_callback,
    scheduled_sampling_max_rate=0.0,  # Task 3: set to 0.10 once Task 1 is confirmed working;
                                      # increase to 0.30 after a stable 0.10 run.
                                      # Keep 0.0 (pure teacher forcing) until optimizer state
                                      # persistence is confirmed working.
)

print("\nTraining process finished.")

## 7. Save the Final Checkpoint

In [ ]:
model.save(CHECKPOINT_DIR)
print(f"Model saved to: {CHECKPOINT_DIR}")

## 8. Quick Inference Test

Load a raw EN clip from the raw dataset and run it through the full pipeline
(WavLM → fine-tuned SpeechT5 transformer → HiFi-GAN) to hear if the model is learning.

In [ ]:
source_lang = "en"
target_lang = "de"
# model.load("checkpoint_epoch_5")
# Load a raw EN sample from the seamless_align dataset for inference.
print("Loading a raw EN sample for inference test...")
raw = dataset_loader.load_data(
    lang=[source_lang],
    split="train",
    dataset=["fleurs"],
    num_samples=1000,
)
sample = raw[source_lang][0]
audio_array = np.array(sample['audio']['array'], dtype=np.float32)
sr = sample['audio']['sampling_rate']

print(f"Input audio: {len(audio_array)/sr:.2f}s @ {sr}Hz")
print("Original EN audio:")
display(Audio(data=audio_array, rate=sr))

# Run the full pipeline: raw audio → WavLM → fine-tuned SpeechT5 → HiFi-GAN → audio
print("\nRunning fine-tuned inference...")
result = model.run_inference(
    audio_array=audio_array,
    sampling_rate=sr,
    threshold=0.25,
    minlenratio=0.0,
    maxlenratio=1.0,
)

out_audio = result['audio']['array']
out_sr    = result['audio']['sampling_rate']

print(f"Output audio: {len(out_audio)/out_sr:.2f}s @ {out_sr}Hz")
print("\nGenerated DE audio (after fine-tuning):")
display(Audio(data=out_audio, rate=out_sr))

## 8a. Task 2 — Inference Parameter Diagnostic

Run three inference configs to diagnose whether the drone is a **stop-token failure (Mode A)**
or a **cross-attention collapse (Mode B)**.

| Observation | Mode | Next step |
|---|---|---|
| Output shortens and starts to sound speech-like with tight parameters | A (stop token) | Continue training; scheduled sampling (Task 3) will further improve |
| Output stays a drone regardless of parameters | B (cross-attention) | Jump to Task 3 immediately |
| Output shortens but remains a drone | Both | Tasks 3 both required |


In [ ]:
import numpy as np
from IPython.display import Audio, display

# ── Task 2: Inference Parameter Diagnostic ──────────────────────────────────
# audio_array and sr must already be defined (run cell 8 / Quick Inference Test first)

configs = [
    {"label": "Default",         "threshold": 0.50, "maxlenratio": 1.2, "minlenratio": 0.0},
    {"label": "Tight stop",      "threshold": 0.25, "maxlenratio": 1.0, "minlenratio": 0.5},
    {"label": "Very tight stop", "threshold": 0.15, "maxlenratio": 0.8, "minlenratio": 0.5},
]

for cfg in configs:
    result = model.run_inference(
        audio_array=audio_array,
        sampling_rate=sr,
        threshold=cfg["threshold"],
        minlenratio=cfg["minlenratio"],
        maxlenratio=cfg["maxlenratio"],
    )
    out = result['audio']['array']
    duration = len(out) / result['audio']['sampling_rate']
    print(f"\n[{cfg['label']}]  duration={duration:.2f}s  "
          f"(threshold={cfg['threshold']}, maxlenratio={cfg['maxlenratio']})")
    display(Audio(data=out, rate=result['audio']['sampling_rate']))

# ── RECORD YOUR DIAGNOSIS HERE ──────────────────────────────────────────────
# Mode A (stop token failure):   output shortens + becomes more speech-like
# Mode B (cross-attention fail):  output stays drone regardless of parameters
# Both:                          output shortens but remains drone
#
# Observed mode: <FILL IN AFTER RUNNING>


## Bonus: Resume Training from a Checkpoint

If training was interrupted, load the saved checkpoint and continue.

In [ ]:
# ── Uncomment to resume from a saved checkpoint ──────────────────────────────
# RESUME_CHECKPOINT = "speecht5_wavlm_en_de_v2"
# model = SpeechT5WavLM()
# model.load(RESUME_CHECKPOINT)
# model.fine_tune(
#     preprocessed_path=PREPROCESSED_PATH,
#     epochs=EPOCHS,
#     learning_rate=LEARNING_RATE,
#     batch_size=BATCH_SIZE,
# )

## 9. Diagnostic Check: Teacher-Forced Forward Pass

This cell runs a standard training forward pass (using teacher forcing) and passes the predicted mel-spectrogram to the vocoder. If it sounds like speech, the cross-attention alignment is working, and the previous mode-collapse was solely due to the inference loop lacking dropout. If it sounds like a continuous drone, cross-attention alignment has failed.

In [ ]:
import torch
from IPython.display import Audio, display

# 1. Load dataset and get one batch for features
from torch.utils.data import DataLoader
from datasets import load_from_disk
from model import SpeechT5WavLMDataset, wavlm_speecht5_collate_fn

paired_ds = load_from_disk(PREPROCESSED_PATH)
train_dataset = SpeechT5WavLMDataset(paired_ds, model.processor, model.target_embeddings)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=wavlm_speecht5_collate_fn)

batch = next(iter(train_loader))
input_values, _, labels, speaker_embeddings = batch

wavlm_features = input_values[0] # (Seq, 768)
speaker_emb = speaker_embeddings[0] # (512,)

print("Starting autoregressive generation (this may take a few seconds)...Array shape:", wavlm_features.shape)

# The infer method handles the encoder bridging, the step-by-step mel generation, 
# and the vocoding all in one go.
audio_arr, pred_mel = model.infer(
    wavlm_hidden_states=wavlm_features, 
    speaker_embeddings=speaker_emb,
    return_spectrogram=True
)

import matplotlib.pyplot as plt
import numpy as np
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

target_mel = labels[0].cpu().numpy()
target_mel[target_mel == -100] = np.nan # Mask padding
im1 = axes[0].imshow(target_mel.T, origin='lower', aspect='auto')
axes[0].set_title("Ground Truth Mel-spectrogram")
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(pred_mel.T, origin='lower', aspect='auto')
axes[1].set_title("Predicted Mel-spectrogram")
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

print("Done! Listen below:")
display(Audio(audio_arr, rate=16000))
